In [1]:
from pathlib import Path
from typing import Optional
import datetime as dt
import sys

import pandas as pd
import requests

from IPython.display import display
from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def detectar_databricks() -> bool:
    try:
        dbutils.secrets
        return True
    except (NameError, AttributeError):
        return False


def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidato in [atual, *atual.parents]:
        if (candidato / "src").is_dir():
            return candidato

    raise FileNotFoundError(
        "Pasta 'src' não encontrada no diretório atual "
        "ou nos diretórios pais."
    )


def exigir_valor(
    valor: Optional[str],
    nome: str,
) -> str:
    valor_normalizado = (
        str(valor).strip()
        if valor is not None
        else ""
    )

    if not valor_normalizado:
        raise RuntimeError(
            f"{nome} não configurado."
        )

    return valor_normalizado


def criar_sessao_http() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=2,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=[
            "GET",
        ],
        raise_on_status=False,
    )

    session = requests.Session()

    session.mount(
        "https://",
        HTTPAdapter(
            max_retries=retry
        ),
    )

    session.headers.update({
        "User-Agent": (
            "Databricks-IBGE-Pipeline/1.0"
        ),
        "Accept": "application/json",
    })

    return session


IS_DATABRICKS = detectar_databricks()
BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )

if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = BASE_DIR / ".env"

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: "
            f"{ENV_PATH}"
        )

    ambiente_carregado = load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )

    if not ambiente_carregado:
        raise RuntimeError(
            f"Não foi possível carregar: "
            f"{ENV_PATH}"
        )

from src.config.secrets import get_secret

AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT",
    ),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY",
    ),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret(
        "AZURE-CONTAINER-BRONZE",
        "AZURE_CONTAINER_BRONZE",
    ),
    "AZURE_CONTAINER_BRONZE",
)

LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "dados"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

azure_credential = AzureNamedKeyCredential(
    name=AZURE_STORAGE_ACCOUNT,
    key=AZURE_STORAGE_KEY,
)

ACCOUNT_URL = (
    f"https://{AZURE_STORAGE_ACCOUNT}"
    ".blob.core.windows.net"
)

blob_service_client = BlobServiceClient(
    account_url=ACCOUNT_URL,
    credential=azure_credential,
)

blob_service_client.get_account_information()

container_client = (
    blob_service_client.get_container_client(
        container=BRONZE_CONTAINER
    )
)

container_client.get_container_properties()

IBGE_MUNICIPIOS_URL = (
    "https://servicodados.ibge.gov.br/"
    "api/v1/localidades/municipios"
)

with criar_sessao_http() as session:
    response = session.get(
        url=IBGE_MUNICIPIOS_URL,
        timeout=(30, 300),
    )

    response.raise_for_status()
    municipios_json = response.json()

if not isinstance(
    municipios_json,
    list,
):
    raise TypeError(
        "A API do IBGE retornou um tipo inesperado: "
        f"{type(municipios_json).__name__}"
    )

if not municipios_json:
    raise ValueError(
        "A API do IBGE retornou uma lista vazia."
    )

df_municipios = pd.json_normalize(
    municipios_json
)

MAPA_COLUNAS = {
    "id": "municipio_id",
    "nome": "municipio_nome",
    "microrregiao.id": "microrregiao_id",
    "microrregiao.nome": "microrregiao_nome",
    "microrregiao.mesorregiao.id": "mesorregiao_id",
    "microrregiao.mesorregiao.nome": "mesorregiao_nome",
    "microrregiao.mesorregiao.UF.id": "estado_id",
    "microrregiao.mesorregiao.UF.sigla": "estado_sigla",
    "microrregiao.mesorregiao.UF.nome": "estado_nome",
    "microrregiao.mesorregiao.UF.regiao.id": "regiao_id",
    "microrregiao.mesorregiao.UF.regiao.sigla": "regiao_sigla",
    "microrregiao.mesorregiao.UF.regiao.nome": "regiao_nome",
    "regiao-imediata.id": "regiao_imediata_id",
    "regiao-imediata.nome": "regiao_imediata_nome",
    "regiao-imediata.regiao-intermediaria.id": (
        "regiao_intermediaria_id"
    ),
    "regiao-imediata.regiao-intermediaria.nome": (
        "regiao_intermediaria_nome"
    ),
    "regiao-imediata.regiao-intermediaria.UF.id": (
        "ri_estado_id"
    ),
    "regiao-imediata.regiao-intermediaria.UF.sigla": (
        "ri_estado_sigla"
    ),
    "regiao-imediata.regiao-intermediaria.UF.nome": (
        "ri_estado_nome"
    ),
    "regiao-imediata.regiao-intermediaria.UF.regiao.id": (
        "ri_regiao_id"
    ),
    "regiao-imediata.regiao-intermediaria.UF.regiao.sigla": (
        "ri_regiao_sigla"
    ),
    "regiao-imediata.regiao-intermediaria.UF.regiao.nome": (
        "ri_regiao_nome"
    ),
}

df_municipios = df_municipios.rename(
    columns=MAPA_COLUNAS
)

COLUNAS_OBRIGATORIAS = [
    "municipio_id",
    "municipio_nome",
]

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS
    if coluna not in df_municipios.columns
]

if colunas_ausentes:
    raise ValueError(
        "Colunas obrigatórias ausentes: "
        f"{colunas_ausentes}"
    )

COLUNAS_DESEJADAS = [
    "municipio_id",
    "municipio_nome",
    "microrregiao_id",
    "microrregiao_nome",
    "mesorregiao_id",
    "mesorregiao_nome",
    "estado_id",
    "estado_sigla",
    "estado_nome",
    "regiao_id",
    "regiao_sigla",
    "regiao_nome",
    "regiao_imediata_id",
    "regiao_imediata_nome",
    "regiao_intermediaria_id",
    "regiao_intermediaria_nome",
    "ri_estado_id",
    "ri_estado_sigla",
    "ri_estado_nome",
    "ri_regiao_id",
    "ri_regiao_sigla",
    "ri_regiao_nome",
]

colunas_disponiveis = [
    coluna
    for coluna in COLUNAS_DESEJADAS
    if coluna in df_municipios.columns
]

df_municipios = df_municipios[
    colunas_disponiveis
].copy()

COLUNAS_NUMERICAS = [
    "municipio_id",
    "microrregiao_id",
    "mesorregiao_id",
    "estado_id",
    "regiao_id",
    "regiao_imediata_id",
    "regiao_intermediaria_id",
    "ri_estado_id",
    "ri_regiao_id",
]

for coluna in COLUNAS_NUMERICAS:
    if coluna in df_municipios.columns:
        df_municipios[coluna] = (
            pd.to_numeric(
                df_municipios[coluna],
                errors="coerce",
           )
            .astype("Int64")
        )

COLUNAS_TEXTO = [
    "municipio_nome",
    "microrregiao_nome",
    "mesorregiao_nome",
    "estado_sigla",
    "estado_nome",
    "regiao_sigla",
    "regiao_nome",
    "regiao_imediata_nome",
    "regiao_intermediaria_nome",
    "ri_estado_sigla",
    "ri_estado_nome",
    "ri_regiao_sigla",
    "ri_regiao_nome",
]

for coluna in COLUNAS_TEXTO:
    if coluna in df_municipios.columns:
        df_municipios[coluna] = (
            df_municipios[coluna]
            .astype("string")
            .str.strip()
        )

INGESTED_AT = dt.datetime.now(
    dt.timezone.utc
)

df_municipios["_ingested_at"] = (
    INGESTED_AT
)

df_municipios["_source"] = (
    IBGE_MUNICIPIOS_URL
)

colunas_ordenacao = [
    coluna
    for coluna in [
        "estado_sigla",
        "municipio_nome",
    ]
    if coluna in df_municipios.columns
]

if colunas_ordenacao:
    df_municipios = (
        df_municipios
        .sort_values(
            by=colunas_ordenacao,
            na_position="last",
        )
        .reset_index(
            drop=True
        )
    )

if df_municipios.empty:
    raise ValueError(
        "DataFrame IBGE Municípios vazio."
    )

if df_municipios[
    "municipio_id"
].isnull().any():
    raise ValueError(
        "Existem municípios sem ID."
    )

if df_municipios[
    "municipio_nome"
].isnull().any():
    raise ValueError(
        "Existem municípios sem nome."
    )

if df_municipios[
    "municipio_id"
].duplicated().any():
    ids_duplicados = (
        df_municipios.loc[
            df_municipios[
                "municipio_id"
            ].duplicated(
                keep=False
            ),
            "municipio_id",
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    raise ValueError(
        "IDs de município duplicados: "
        f"{ids_duplicados[:20]}"
    )

DATE_SUFFIX = INGESTED_AT.strftime(
    "%Y-%m-%d"
)

BLOB_NAME = (
    f"{DATE_SUFFIX}_"
    "ibge_municipios.parquet"
)

PARQUET_FILE = (
    LOCAL_BRONZE_PATH
    / BLOB_NAME
)

df_municipios.to_parquet(
    PARQUET_FILE,
    index=False,
    engine="pyarrow",
)

if not PARQUET_FILE.is_file():
    raise FileNotFoundError(
        f"Parquet não criado: "
        f"{PARQUET_FILE}"
    )

TAMANHO_LOCAL = (
    PARQUET_FILE.stat().st_size
)

if TAMANHO_LOCAL <= 0:
    raise RuntimeError(
        f"Parquet criado sem conteúdo: "
        f"{PARQUET_FILE}"
    )

blob_client = (
    container_client.get_blob_client(
        blob=BLOB_NAME
    )
)

with PARQUET_FILE.open(
    mode="rb"
) as arquivo:
    blob_client.upload_blob(
        data=arquivo,
        overwrite=True,
    )

blob_properties = (
    blob_client.get_blob_properties()
)

if blob_properties.size != TAMANHO_LOCAL:
    raise RuntimeError(
        "O tamanho enviado não corresponde "
        "ao arquivo local. "
        f"Local: {TAMANHO_LOCAL} bytes. "
        f"Azure: {blob_properties.size} bytes."
    )

print(
    "========== RESULTADO =========="
)

print(
    "✅ Ambiente:",
    "Databricks"
    if IS_DATABRICKS
    else "local",
)

print(
    f"✅ Raiz do projeto: "
    f"{BASE_DIR}"
)

print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Container: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Municípios retornados: "
    f"{len(municipios_json)}"
)

print(
    f"✅ Municípios carregados: "
    f"{len(df_municipios)}"
)

print(
    f"✅ Municípios únicos: "
    f"{df_municipios['municipio_id'].nunique()}"
)

if "estado_sigla" in df_municipios.columns:
    print(
        f"✅ Estados únicos: "
        f"{df_municipios['estado_sigla'].nunique()}"
    )

if "regiao_sigla" in df_municipios.columns:
    print(
        f"✅ Regiões únicas: "
        f"{df_municipios['regiao_sigla'].nunique()}"
    )

print(
    f"✅ Colunas: "
    f"{len(df_municipios.columns)}"
)

print(
    f"✅ Arquivo local: "
    f"{PARQUET_FILE.resolve()}"
)

print(
    f"✅ Blob Azure: "
    f"{BLOB_NAME}"
)

print(
    f"✅ Tamanho local: "
    f"{TAMANHO_LOCAL / 1024:.2f} KB"
)

print(
    f"✅ Tamanho Azure: "
    f"{blob_properties.size / 1024:.2f} KB"
)

print(
    f"✅ Ingestão UTC: "
    f"{INGESTED_AT.isoformat()}"
)

print(
    "✅ Parquet local e blob Azure "
    "validados com sucesso."
)

display(
    df_municipios
    if IS_DATABRICKS
    else df_municipios.head(2)
)

========== RESULTADO ==========
✅ Ambiente: local
✅ Raiz do projeto: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28
✅ Storage Account: stfiaptechchallenge2iast
✅ Container: bronze
✅ Municípios retornados: 5571
✅ Municípios carregados: 5571
✅ Municípios únicos: 5571
✅ Estados únicos: 27
✅ Regiões únicas: 5
✅ Colunas: 24
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-19_ibge_municipios.parquet
✅ Blob Azure: 2026-10-19_ibge_municipios.parquet
✅ Tamanho local: 174.71 KB
✅ Tamanho Azure: 174.71 KB
✅ Ingestão UTC: 2026-10-19T21:00:57.725696+00:00
✅ Parquet local e blob Azure validados com sucesso.


,municipio_id,municipio_nome,microrregiao_id,microrregiao_nome,mesorregiao_id,mesorregiao_nome,estado_id,estado_sigla,estado_nome,regiao_id,...,regiao_intermediaria_id,regiao_intermediaria_nome,ri_estado_id,ri_estado_sigla,ri_estado_nome,ri_regiao_id,ri_regiao_sigla,ri_regiao_nome,_ingested_at,_source
0,1200013,Acrelândia,12004,Rio Branco,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,2026-10-19 21:00:57.725696+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
1,1200054,Assis Brasil,12005,Brasiléia,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,2026-10-19 21:00:57.725696+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
